# Importing Libraries

In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
from PIL import Image

In [3]:
model = tf.keras.models.load_model('food_detection_model.h5')

In [4]:
class_names = [
'caesar_salad',
'chocolate_cake',
'edamame',
'french_fries',
'fried_rice',
'greek_salad',
'grilled_salmon',
'hamburger',
'ice_cream',
'omelette',
'pizza',
'sushi'
]

In [5]:
def predict_food(image_path):

    img = Image.open(image_path).resize((224,224))
    img_array = np.array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)

    predicted_class = class_names[np.argmax(prediction)]

    return predicted_class

In [6]:
nutrition_df = pd.read_csv("nutrition_dataset.csv")

nutrition_df.head()

,food,Caloric Value,Fat,Saturated Fats,Sugars,Sodium
0,cream cheese,51,5.0,2.9,0.500,0.016
1,neufchatel cheese,215,19.4,10.9,2.700,0.300
2,requeijao cremoso light catupiry,49,3.6,2.3,3.400,0.000
3,ricotta cheese,30,2.0,1.3,0.091,0.017
4,cream cheese low fat,30,2.3,1.4,0.900,0.046


In [7]:
food_mapping = {
    "pizza": "pizza",
    "hamburger": "hamburger",
    "french_fries": "french fries",
    "fried_rice": "fried rice",
    "chocolate_cake": "chocolate cake",
    "ice_cream": "ice cream",
    "caesar_salad": "caesar salad",
    "greek_salad": "greek salad",
    "edamame": "edamame",
    "grilled_salmon": "salmon",
    "sushi": "sushi",
    "omelette": "omelet"
}

In [8]:
def map_food_name(predicted_food):
    return food_mapping.get(predicted_food, predicted_food)

In [25]:
def get_nutrition(food_name):

    mapped_food = map_food_name(food_name)

    rows = nutrition_df[
        nutrition_df["food"].str.lower().str.contains(mapped_food.lower())
    ]

    if rows.empty:
        return None

    return rows.mean(numeric_only=True)

In [26]:
SAT_FAT_LIMIT = 8
SODIUM_LIMIT = 500
SUGAR_LIMIT = 10
CALORIE_LIMIT = 300

In [32]:
def calculate_heart_risk(nutrition):

    score = 0
    reasons = []

    sodium_mg = nutrition["Sodium"] * 1000

    if nutrition["Saturated Fats"] > SAT_FAT_LIMIT:
        score += 1
        reasons.append("High saturated fat")

    if sodium_mg > SODIUM_LIMIT:
        score += 1
        reasons.append("High sodium")

    if nutrition["Sugars"] > SUGAR_LIMIT:
        score += 1
        reasons.append("High sugar")

    if nutrition["Caloric Value"] > CALORIE_LIMIT:
        score += 1
        reasons.append("High calories")

    if score == 0:
        risk = "Low"
    elif score == 1:
        risk = "Moderate"
    else:
        risk = "High"

    return risk, reasons

In [48]:
def analyze_food_image(image_path):

    food = predict_food(image_path)

    nutrition = get_nutrition(food)

    if nutrition is None:
        print("Food not found in nutrition dataset")
        return

    risk, reasons = calculate_heart_risk(nutrition)

    print("Food detected:", food)

    print("\nNutrition Information")
    print("---------------------")

    print("Calories:", f"{nutrition['Caloric Value']:.2f}", "kcal")
    print("Fat:", f"{nutrition['Fat']:.2f}", "g")
    print("Saturated Fat:", f"{nutrition['Saturated Fats']:.2f}", "g")
    print("Sugar:", f"{nutrition['Sugars']:.2f}", "g")
    print("Sodium:", f"{nutrition['Sodium']:.2f}", "mg")

    print("\nHeart Health Risk:", risk)
    
    if risk == "Low":
        print("This food is generally considered heart-healthy based on its nutritional content.")
    else:
        print("\nReasons:")
        for r in reasons:
            print("•", r)

In [54]:
analyze_food_image("fries.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Food detected: french_fries

Nutrition Information
---------------------
Calories: 164.80 kcal
Fat: 7.66 g
Saturated Fat: 1.21 g
Sugar: 0.19 g
Sodium: 0.12 mg

Heart Health Risk: Low
This food is generally considered heart-healthy based on its nutritional content.
